# 027 — Redes bayesianas e independencia condicional

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** (a) **Sí**: el único camino R→A←T tiene un colisionador (A) no observado. (b) **No**: observar A abre el colisionador. (c) **Sí**: el camino J←A→M es un fork bloqueado por A. (d) **Sí**: el camino R→A→J es una cadena bloqueada por A. (e) **No**: M es descendiente del colisionador A; observarlo también abre el camino R→A←T.

**E2.** Red: `1(R) + 1(T) + 4(A|R,T) + 2(J|A) + 2(M|A) = 10`. Conjunta plena: `2⁵−1 = 31`. Con un tercer padre de A: la CPT de A pasa de 4 a 8 filas → 14 parámetros; el costo crece exponencialmente con los padres, no con el total de nodos.

**E3.** `P(A) = 0.95·0.01·0.02 + 0.94·0.01·0.98 + 0.29·0.99·0.02 + 0.001·0.99·0.98 ≈ 0.0161`. `P(J) = 0.9·0.0161 + 0.05·0.9839 ≈ 0.0637`. `P(R,J) = Σ_t Σ_a P(R)P(t)P(a|R,t)P(J|a) ≈ 0.01·(0.9·0.9402 + 0.05·0.0598) ≈ 0.00849`. `P(R|J) ≈ 0.00849/0.0637 ≈ 0.133`: la llamada de Juan multiplica la creencia en robo ×13.

**E4.** `P(R|A) = P(R,A)/P(A) ≈ 0.0094/0.0161 ≈ 0.583` y `P(R|A,T) = 0.95·0.01/(0.95·0.01+0.29·0.99) ≈ 0.032`. **Baja** ×18: el terremoto ya explica la alarma, así que el robo pierde soporte — dependencia inducida entre padres al observar el colisionador.


In [ ]:
result = run_lab("probability", seed=27)
assert result["kind"] == "probability"
assert result["evidence"]
show(result)


In [ ]:
from itertools import product
pR, pT = 0.01, 0.02
pA = {(1,1):0.95, (1,0):0.94, (0,1):0.29, (0,0):0.001}
pJ = {1:0.90, 0:0.05}

def joint(r, t, a, j):
    p = (pR if r else 1-pR) * (pT if t else 1-pT)
    pa = pA[(r,t)]
    p *= pa if a else 1-pa
    pj = pJ[a]
    return p * (pj if j else 1-pj)

worlds = list(product([1,0], repeat=4))
P = lambda f: sum(joint(*w) for w in worlds if f(*w))
pJ1 = P(lambda r,t,a,j: j==1)
print("P(A)      =", round(P(lambda r,t,a,j: a==1), 4))
print("P(R|J)    =", round(P(lambda r,t,a,j: r==1 and j==1)/pJ1, 3))
pA1 = P(lambda r,t,a,j: a==1)
print("P(R|A)    =", round(P(lambda r,t,a,j: r==1 and a==1)/pA1, 3))
pAT = P(lambda r,t,a,j: a==1 and t==1)
print("P(R|A,T)  =", round(P(lambda r,t,a,j: r==1 and a==1 and t==1)/pAT, 3))


## Reflexión

1. En la red del laboratorio, ¿qué independencia condicional concreta reduce el número de parámetros? Escríbela en notación `X ⊥ Y | Z`.
2. ¿Por qué observar un colisionador *crea* dependencia entre sus padres? Explica *explaining away* con las variables de esta clase.
3. ¿Qué pasa con la factorización si añades un arco que forme un ciclo? ¿Por qué la definición exige un DAG?
